In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import matplotlib.pyplot as plt

In [134]:
def format_data(
    data: pd.DataFrame,
    inplace: bool = False
) -> pd.DataFrame:

    res = data if inplace else data.copy()

    # Cambio de nombre de columna genero del lector
    columnas = list(res.columns)
    indices = [i for i, col in enumerate(columnas) if str(col).startswith('genero')]
    columnas[indices[1]] = 'sexo'
    res.columns = columnas

    # Formato de fecha
    if 'fecha' in res.columns:
        res['fecha'] = pd.to_datetime(res['fecha'], format='%d-%m-%Y', errors='coerce')
    res['nacimiento'] = pd.to_numeric(res['nacimiento'], errors='coerce').astype('Int64')
    res['anio_edicion'] = pd.to_numeric(res['anio_edicion'], errors='coerce').astype('Int64')

    return res

In [94]:
def extraer_ciudad_pais(
    df: pd.DataFrame,
    col_origen: str = "vive_en",
    col_ciudad: str = "vive_en_ciudad",
    col_pais: str = "vive_en_pais",
    inplace: bool = False,
) -> pd.DataFrame:
    res = df if inplace else df.copy()

    if res.empty or col_origen not in res.columns:
        res[col_ciudad] = None
        res[col_pais] = None
        return res

    # 1. Limpieza inicial tratando nulos y vacíos
    texto_limpio = res[col_origen].fillna("").astype(str).str.strip()

    # 2. Split por guión
    partes = texto_limpio.str.split("-", n=1, expand=True)

    # 3. Si ninguna fila tenía '-', creamos la columna 1 con None para evitar KeyError
    if 1 not in partes.columns:
        partes[1] = None

    # 4. Asignación condicional
    tiene_guion = partes[1].notna()

    ciudad = np.where(tiene_guion, partes[0].str.strip(), None)
    pais = np.where(tiene_guion, partes[1].str.strip(), partes[0].str.strip())

    # 5. Forzar None si los valores resultantes quedaron vacíos
    res[col_ciudad] = np.where((ciudad == "") | (ciudad == "None"), None, ciudad)
    res[col_pais] = np.where((pais == "") | (pais == "None"), None, pais)

    return res

In [124]:
conexion = sqlite3.connect('data.db')
cursor = conexion.cursor()

# Lista todas las tablas dentro de la base de datos
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tablas = cursor.fetchall()

print("Tus tablas son:", tablas)
conexion.close()

Tus tablas son: [('libros',), ('lectores',), ('interacciones',)]


In [125]:
# Crear conexión y leer la consulta
conexion = sqlite3.connect('data.db')
df = pd.read_sql_query("SELECT b.*, c.*, a.fecha, a.rating FROM interacciones AS a INNER JOIN libros AS b ON a.id_libro = b.id_libro INNER JOIN lectores as c on a.id_lector = c.id_lector", conexion)

conexion.close()

In [126]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 461074 entries, 0 to 461073
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   id_libro      461074 non-null  str   
 1   titulo        461074 non-null  str   
 2   autor         461070 non-null  str   
 3   genero        461074 non-null  str   
 4   editorial     461048 non-null  str   
 5   anio_edicion  461074 non-null  object
 6   isbn          461074 non-null  object
 7   resumen       461017 non-null  str   
 8   img_src       461074 non-null  str   
 9   id_lector     461074 non-null  str   
 10  nombre        461074 non-null  str   
 11  genero        461074 non-null  str   
 12  vive_en       461074 non-null  str   
 13  nacimiento    461074 non-null  str   
 14  fecha         461074 non-null  str   
 15  rating        461074 non-null  int64 
dtypes: int64(1), object(2), str(13)
memory usage: 56.3+ MB


In [127]:
df = format_data(df)
df = extraer_ciudad_pais(df)

In [128]:
df_filtrado = df[df.groupby('id_lector')['rating'].transform('count') > 20]

# Separo test con los ÚLTIMOS 20 registros de cada lector
df_test = df_filtrado.sort_values(by=['id_lector', 'fecha']).groupby('id_lector').tail(20)
df_train = df_filtrado.drop(df_test.index)

print(df_train.shape)
print(df_test.shape)

(346890, 18)
(76020, 18)


In [212]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV

# 1. Preparar features y target
drop_columns = ['rating', 'id_libro', 'titulo', 'isbn', 'resumen', 'img_src', 'id_lector', 'nombre', 'fecha', 'vive_en']

X_train = df_train.drop(columns=drop_columns, errors='ignore').copy()
y_train = df_train['rating']
X_test = df_test.drop(columns=drop_columns, errors='ignore').copy()
y_test = df_test['rating']

# 2. Convertir columnas de texto a tipo category
cols_texto = X_train.select_dtypes(include=['object', 'string']).columns
for col in cols_texto:
    X_train[col] = X_train[col].astype('category')

# 3. Definir grilla acotada
param_grid = {
    'n_estimators': [250, 500, 1000],
    'learning_rate': [0.03, 0.1],
    'max_depth': [5, 10, 12],
    'min_child_samples': [20],
    'colsample_bytree': [0.8],
    'subsample': [0.8]
}

# 4. Configurar estimador y Grid Search
lgb_base = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

grid_search = GridSearchCV(
    estimator=lgb_base,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

# 5. Ejecutar búsqueda
grid_search.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor RMSE en CV: {-grid_search.best_score_:.4f}")

# Modelo óptimo listo para usar en la inferencia
model = grid_search.best_estimator_

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Mejores hiperparámetros: {'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_samples': 20, 'n_estimators': 1000, 'subsample': 0.8}
Mejor RMSE en CV: 1.6972


In [214]:
# 1. Crear el DataFrame con los nombres de las variables y sus valores
feature_imp = pd.DataFrame({
    'variable': X_train.columns,
    'gain': model.booster_.feature_importance(importance_type='gain'),
    'split': model.booster_.feature_importance(importance_type='split')
})

# 2. Ordenar de mayor a menor por 'gain'
feature_imp = feature_imp.sort_values(by='gain', ascending=False).reset_index(drop=True)

# Mostrar el Top 10 en texto
print(feature_imp.head(10))

         variable          gain  split
0           autor  1.656820e+06   4219
1  vive_en_ciudad  9.652166e+05   4438
2       editorial  4.840463e+05   3664
3          genero  3.634322e+05   1672
4      nacimiento  2.097243e+05   6433
5    vive_en_pais  1.368268e+05    536
6    anio_edicion  9.976301e+04   4485
7            sexo  4.750815e+04   2157


In [215]:
from sklearn.metrics import mean_squared_error

# Columnas categóricas
cols_texto = X_test.select_dtypes(include=['object', 'str']).columns

for col in cols_texto:
    if hasattr(X_train[col], 'cat'):
        X_test[col] = (
            X_test[col]
            .astype('category')
            .cat.set_categories(X_train[col].cat.categories)
        )
    else:
        X_test[col] = X_test[col].astype('category')

preds_test = model.predict(X_test)
print("RMSE:", mean_squared_error(y_test, preds_test))

RMSE: 3.0390745098750456


In [216]:
conexion = sqlite3.connect('data.db')

# Lista para acumular los top 20 de cada lector
recomendaciones_por_lector = []

df_todos_libros = pd.read_sql_query("SELECT * FROM libros", conexion)
df_lectores = pd.read_sql_query("SELECT * FROM lectores", conexion)

# Cerrar conexión al finalizar
conexion.close()

for lector in df_test['id_lector'].unique().tolist():
    train_lector = df_train[df_train['id_lector'] == lector]
    generos_vistos = train_lector['genero'].unique()
    libros_vistos_train = set(train_lector['id_libro'])

    # Filtrar libros candidatos
    df_libros = df_todos_libros[
        df_todos_libros['genero'].isin(generos_vistos) &
        (~df_todos_libros['id_libro'].isin(libros_vistos_train))
    ]

    if df_libros.empty:
        print(f"el lector {lector} no califico ningun libro aun.")
        continue

    df_lector = df_lectores[df_lectores['id_lector'] == lector]

    if df_lector.empty:
        print(f"{lector} no existe")
        continue

    # Transformaciones
    if not df_lector.empty:
        df_test_a_predecir = df_libros.merge(df_lector, how="cross", suffixes=("", "_"))
        df_test_a_predecir = format_data(df_test_a_predecir)
        df_test_a_predecir = extraer_ciudad_pais(df_test_a_predecir)

    X_test_pred = df_test_a_predecir[X_train.columns].copy()

    # Columnas categóricas
    cols_texto = X_test_pred.select_dtypes(include=['object', 'str']).columns

    for col in cols_texto:
        if hasattr(X_train[col], 'cat'):
            X_test_pred[col] = (
                X_test_pred[col]
                .astype('category')
                .cat.set_categories(X_train[col].cat.categories)
            )
        else:
            X_test_pred[col] = X_test_pred[col].astype('category')

    # Predicción y ranking
    df_test_a_predecir["score_pred"] = model.predict(X_test_pred)

    top_20_lector = (
        df_test_a_predecir[
            ["id_lector", "id_libro", "titulo", "genero", "score_pred"]
        ]
        .sort_values(by="score_pred", ascending=False)
        .head(20)
        .copy()
    )

    top_20_lector["ranking"] = range(1, len(top_20_lector) + 1)

    recomendaciones_por_lector.append(top_20_lector)

df_test_pred = pd.concat(recomendaciones_por_lector, ignore_index=True)

In [217]:
df_test_pred['score_pred'].describe()

count    76020.000000
mean         9.353306
std          0.362017
min          6.888678
25%          9.215781
50%          9.362685
75%          9.547218
max         11.481253
Name: score_pred, dtype: float64

In [218]:
df_test_a_predecir.info()

<class 'pandas.DataFrame'>
RangeIndex: 26985 entries, 0 to 26984
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_libro        26985 non-null  str    
 1   titulo          26985 non-null  str    
 2   autor           26984 non-null  str    
 3   genero          26985 non-null  str    
 4   editorial       26980 non-null  str    
 5   anio_edicion    26959 non-null  Int64  
 6   isbn            26985 non-null  object 
 7   resumen         26977 non-null  str    
 8   img_src         26985 non-null  str    
 9   id_lector       26985 non-null  str    
 10  nombre          26985 non-null  str    
 11  sexo            26985 non-null  str    
 12  vive_en         26985 non-null  str    
 13  nacimiento      26985 non-null  Int64  
 14  vive_en_ciudad  26985 non-null  str    
 15  vive_en_pais    26985 non-null  str    
 16  score_pred      26985 non-null  float64
dtypes: Int64(2), float64(1), object(1), str(13

In [219]:
import numpy as np
from sklearn.metrics import ndcg_score

def compute_ndcg_sklearn(df_test, df_test_pred, item_col='id_libro', user_col='id_lector', k=20):
    """
    Calcula NDCG@K usando la función de scikit-learn.
    """
    ndcg_list = []

    # Lectores presentes en ambos sets
    lectores = set(df_test_pred[user_col].unique()).intersection(set(df_test[user_col].unique()))

    for lector in lectores:
        # 1. Ítems recomendados por el modelo (Top-K) con sus scores predichos
        preds_user = df_test_pred[df_test_pred[user_col] == lector][[item_col, 'score_pred']]

        # 2. Ítems reales que el usuario calificó en test con sus ratings verdaderos
        test_user = df_test[df_test[user_col] == lector][[item_col, 'rating']]

        # Si el usuario no tiene calificaciones positivas en test, se omite
        if test_user['rating'].sum() == 0:
            continue

        # 3. Unión completa de ítems (Recomendados U Reales de Test)
        # Esto permite que sklearn conozca tanto lo que predijiste como el ideal real
        df_union = pd.merge(preds_user, test_user, on=item_col, how='outer')

        # Rellenar:
        # - Si el ítem no fue recomendado: score predicho muy bajo (-inf) para que quede al fondo
        # - Si el ítem no estaba en test: relevancia real = 0
        df_union['score_pred'] = df_union['score_pred'].fillna(0.0)
        df_union['rating'] = df_union['rating'].fillna(0.0)

        y_true = df_union['rating'].values.reshape(1, -1)
        y_score = df_union['score_pred'].values.reshape(1, -1)

        # 4. Calcular con scikit-learn
        # Nota: si hay menos ítems en la unión que k, k se ajusta automáticamente
        k_eval = min(k, y_true.shape[1])
        score = ndcg_score(y_true, y_score, k=k_eval)
        ndcg_list.append(score)

    return np.mean(ndcg_list) if ndcg_list else 0.0

# Evaluación
k = 20
mean_ndcg = compute_ndcg_sklearn(df_test, df_test_pred, item_col='id_libro', user_col='id_lector', k=k)
print(f"Mean NDCG@{k}: {mean_ndcg:.5f}")

Mean NDCG@20: 0.00056


In [220]:
df_ejemplo = pd.read_csv("ejemplo.csv")
lectores_a_predecir = df_ejemplo['id_lector'].drop_duplicates()

In [221]:
lectores_a_predecir.nunique()

832

In [222]:
conexion = sqlite3.connect('data.db')

# Lista para acumular los top 20 de cada lector
recomendaciones_por_lector = []

for lector in lectores_a_predecir.tolist():
    # 1. Libros de géneros leídos, excluyendo los que ya interactuó
    query_libros = """
    SELECT *
    FROM libros
    WHERE libros.genero IN (
        SELECT DISTINCT l.genero
        FROM interacciones i
        JOIN libros l ON i.id_libro = l.id_libro
        WHERE i.id_lector = ?
    )
    AND libros.id_libro NOT IN (
        SELECT id_libro
        FROM interacciones
        WHERE id_lector = ?
    );
    """
    df_libros = pd.read_sql_query(
        query_libros, conexion, params=[lector, lector]
    )

    if df_libros.empty:
        print(f"el lector {lector} no califico ningun libro aun.")
        continue

    # 2. Consulta de lector
    query_lectores = """
    SELECT *
    FROM lectores
    WHERE id_lector = ?
    """
    df_lector = pd.read_sql_query(query_lectores, conexion, params=[lector])

    if df_lector.empty:
        print(f"{lector} no existe")
        continue

    df_a_predecir = df_libros.merge(df_lector, how="cross", suffixes=("", "_"))
    df_a_predecir = format_data(df_a_predecir)
    df_a_predecir = extraer_ciudad_pais(df_a_predecir)

    X_pred = df_a_predecir[X_train.columns].copy()

    # Columnas categóricas
    cols_texto = X_pred.select_dtypes(include=['object', 'str']).columns

    for col in cols_texto:
        if hasattr(X_train[col], 'cat'):
            X_pred[col] = (
                X_pred[col]
                .astype('category')
                .cat.set_categories(X_train[col].cat.categories)
            )
        else:
            X_pred[col] = X_pred[col].astype('category')

    # Predicción y ranking
    df_a_predecir["score_pred"] = model.predict(X_pred)

    top_20_lector = (
        df_a_predecir[
            ["id_lector", "id_libro", "titulo", "genero", "score_pred"]
        ]
        .sort_values(by="score_pred", ascending=False)
        .head(20)
        .copy()
    )

    top_20_lector["ranking"] = range(1, len(top_20_lector) + 1)

    recomendaciones_por_lector.append(top_20_lector)

# Cerrar conexión al finalizar
conexion.close()

df_pred = pd.concat(recomendaciones_por_lector, ignore_index=True)

el lector 102893014085601872783 no califico ningun libro aun.
el lector 103084137542939876557 no califico ningun libro aun.
el lector 104186425412399255991 no califico ningun libro aun.
el lector 114281238546612894708 no califico ningun libro aun.
el lector 115834876939320404006 no califico ningun libro aun.
el lector 115884316008551273059 no califico ningun libro aun.
el lector 116044910452339432768 no califico ningun libro aun.
el lector acastrop no califico ningun libro aun.
alpasil no existe
el lector carlosecheve no califico ningun libro aun.
el lector diegodacosta no califico ningun libro aun.
el lector eduardexhp no califico ningun libro aun.
el lector india no califico ningun libro aun.
el lector pequod no califico ningun libro aun.
el lector raf86 no califico ningun libro aun.
el lector tommycaceres no califico ningun libro aun.
el lector victor27 no califico ningun libro aun.


In [223]:
df_pred['id_lector'].nunique()

815

In [224]:
df_pred[['id_lector', 'id_libro']]

,id_lector,id_libro
0,05-03-1970,el-falucho-colombo
1,05-03-1970,el-bergantin-penelope
2,05-03-1970,la-fragata-sirena
3,05-03-1970,el-navio-triunfante
4,05-03-1970,el-navio-principe-de-asturias-trafalgar
...,...,...
16295,zapy46,el-poder-titan-2
16296,zapy46,tom-gates-planes-geniales-o-no
16297,zapy46,deidad
16298,zapy46,tom-gates-los-lobozombis-son-geniales-y-punto


In [225]:
lectores_a_predecir.nunique()

832

In [226]:
# 1. Identificar los lectores que faltan
lectores_procesados = set(df_pred["id_lector"].unique())
lectores_totales = set(lectores_a_predecir)
faltantes = list(lectores_totales - lectores_procesados)

print(f"Completando {len(faltantes)} lectores con df_ejemplo...")

# 2. Generar las filas de respaldo para cada lector faltante
filas_fallback = []

for lector in faltantes:
    df_temp = df_ejemplo[df_ejemplo['id_lector'] == lector]

    filas_fallback.append(df_temp)

# 3. Consolidar el bloque de faltantes
df_relleno = pd.concat(filas_fallback, ignore_index=True)

# 4. Unir con las predicciones principales asegurando las mismas columnas
cols_comunes = [col for col in df_pred.columns if col in df_relleno.columns]

df_pred_final = (
    pd.concat([df_pred, df_relleno], ignore_index=True)
    .sort_values(by=["id_lector", "ranking"])
    .reset_index(drop=True)
)

# 5. Verificación final
print(f"Total lectores únicos: {df_pred_final['id_lector'].nunique()}")
print(f"Total registros: {len(df_pred_final)} (deberían ser {len(lectores_a_predecir) * 20})")

Completando 17 lectores con df_ejemplo...
Total lectores únicos: 832
Total registros: 16640 (deberían ser 16640)


In [227]:
df_pred_final[['id_lector', 'id_libro']].to_csv("submission.csv", index=False)

In [228]:
df['autor'].nunique()

20685

In [229]:
df_test_pred['score_pred'].max()

np.float64(11.481253389032897)

In [101]:
# Cuenta los géneros únicos por id_lector y genera el resumen estadístico
# df.groupby("id_lector")["genero"].nunique().describe()